In [1]:
import pandas as pd

from DWG_to_Excel_ETL import *
import os
import shutil
import numpy as np


dwg_folder = r"C:\Users\Usuario\Desktop\z_tilt"
temp_folder = r"C:\Users\Usuario\Desktop\z_tilt\temp"
output_folder = r"C:\Users\Usuario\Desktop\z_tilt\output"

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "teste2.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "teste2.dxf")

debug_dxf_coordinates(dxf_file)

data_ELM = extract_data_from_dxf(dxf_file)
print("ELM Data Loaded")

os.remove(dxf_file)
os.remove(r"C:\Users\Usuario\Desktop\z_tilt\temp\teste2.dwg")

ELM Data Loaded


In [2]:
ELM_df = pd.DataFrame(data_ELM)
print(ELM_df.head())
print(ELM_df.info())
print(ELM_df.describe())

      type layer handle      center_x       center_y  major_radius
0  ELLIPSE     0  54B99  34748.694970  128866.812284           3.0
1  ELLIPSE     0  54B9A  34742.394970  128866.812284           3.0
2  ELLIPSE     0  54B9B  34742.394970  128819.305981           3.0
3  ELLIPSE     0  54B9C  34748.695128  128690.566626           1.0
4  ELLIPSE     0  54B9D  34742.395128  128690.565250           1.0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6819 entries, 0 to 6818
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   type          6819 non-null   object 
 1   layer         6819 non-null   object 
 2   handle        6819 non-null   object 
 3   center_x      6819 non-null   float64
 4   center_y      6819 non-null   float64
 5   major_radius  6819 non-null   float64
dtypes: float64(3), object(3)
memory usage: 319.8+ KB
None
           center_x       center_y  major_radius
count   6819.000000    6819.000000   681

In [3]:
ELM_df['new_name'] = np.where(
    ELM_df['major_radius'] == 1,
    '1x32p',
    np.where(
        ELM_df['major_radius'] == 2,
        '1x64p',
        np.where(
            ELM_df['major_radius'] == 3,
            '1x96p_ponta',
            '1x96p_meio'
        )
    ),
)


print(ELM_df['new_name'].value_counts())
print(ELM_df['layer'].value_counts())

new_name
1x96p_ponta    3363
1x96p_meio     1679
1x64p          1114
1x32p           663
Name: count, dtype: int64
layer
0    6819
Name: count, dtype: int64


In [4]:
ELM_df['Script'] = ELM_df.apply(lambda row:
                       f"._insert {row['new_name']} {abs(row['center_x'])},{abs(row['center_y'])},0 1 1 0", axis=1)



print(ELM_df['Script'])

with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Fixing Z\Script blocks.scr", 'w') as f:
    for i in ELM_df['layer'].unique():
        mask = ELM_df['layer'] == i

        if mask.any():
            f.write(f'(command-s "._layer" "set" "{i}" "")\n')
            f.write('\n'.join(ELM_df[mask]['Script'].astype(str)) + '\n')

0       ._insert 1x96p_ponta 34748.69497000487,128866....
1       ._insert 1x96p_ponta 34742.39497000487,128866....
2       ._insert 1x96p_ponta 34742.39497000486,128819....
3       ._insert 1x32p 34748.69512827332,128690.566626...
4       ._insert 1x32p 34742.39512827332,128690.565249...
                              ...                        
6814    ._insert 1x32p 34255.18614999999,126018.935933...
6815    ._insert 1x32p 34248.88614999999,126018.944696...
6816    ._insert 1x32p 34242.58614999998,126018.951730...
6817    ._insert 1x32p 34236.28614999996,126018.957103...
6818    ._insert 1x32p 34229.98614999998,126018.958701...
Name: Script, Length: 6819, dtype: object
